In [ ]:
# PROPOSITO: dejar el codigo del repo disponible en este servidor de Colab.
# Es la misma celda que abre 00_arranque, para que este cuaderno pueda usarse solo.
# ESPERADO: sin errores. Recuerda hacer push antes si cambiaste codigo en tu PC.
import os
import sys

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "03DataCollectionAndEnvironmentSetup"

if not os.path.exists("/content/repo"):
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo pull

%cd /content/repo
!pip install -q -r requirements.txt
sys.path.insert(0, "/content/repo/src")

In [ ]:
# PROPOSITO: si el servidor es nuevo, /content/datos esta vacio y los datos
# se reconstruyen. Si ya estan, esta celda no hace nada.
# ESPERADO: si los datos existian, no imprime nada. Si no, muestra la ingesta
# (unos 12 minutos por dia, una linea de resumen por dia).
from pathlib import Path
from ptcg_wp.colab import setup_kaggle

if not list(Path("/content/datos").glob("estados_*.parquet")):
    setup_kaggle()
    !PYTHONPATH=src python -m ptcg_wp.ingest --config configs/dias.yaml --raw /content/raw --out /content/datos

In [ ]:
# PROPOSITO: cargar todas las tablas de estados y separar X (caracteristicas) e y (etiqueta).
# - feature_columns quita las columnas meta y falla si se cuela una prohibida (defensa contra fuga).
# - Se excluyen las filas de preparacion (setup == 1, turno 0): premios sin repartir y tablero vacio.
# ESPERADO: 61 caracteristicas; unas 755 mil filas por dia fuera de setup; prevalencia
# cercana a 0.54 (mayor que 0.5 porque el ganador tiene mas pasos propios).
import numpy as np
import pandas as pd
from pathlib import Path
from ptcg_wp.features import feature_columns, SENTINEL

d = Path("/content/datos")
states = pd.concat(
    (pd.read_parquet(p) for p in sorted(d.glob("estados_*.parquet"))), ignore_index=True
)
feats = feature_columns(states.columns)
mask = states["setup"] == 0
X, y = states.loc[mask, feats], states.loc[mask, "y"]
print(len(feats), "caracteristicas |", len(X), "filas fuera de setup | prevalencia:", y.mean().round(3))

In [ ]:
# PROPOSITO: ver el rango de cada caracteristica, cuantas veces falta el dato
# (valor centinela -1) y si alguna es constante (no aporta informacion).
# Las columnas diff_ se excluyen del conteo de centinela porque su -1 es una diferencia real.
# ESPERADO:
# - pct_centinela mayor que 0 solo en columnas active_* (cuando no hay Pokemon activo);
#   debe ser bajo, idealmente menor a 1 %.
# - constante: debe ser False en casi todas. Las que salgan True se pueden eliminar.
# - Los minimos y maximos deben tener sentido (premios entre 0 y 6, hp positivo, etc.).
base = [c for c in feats if not c.startswith("diff_")]
resumen = pd.DataFrame({
    "min": X.min(),
    "max": X.max(),
    "media": X.mean().round(3),
    "pct_centinela": (X[base] == SENTINEL).mean().mul(100).round(2),
    "constante": X.nunique() <= 1,
})
resumen

In [ ]:
# PROPOSITO: buscar datos que no pueden existir en un juego real. Cada numero cuenta
# filas con un valor imposible; si no es 0, hay un error en la extraccion o en los datos.
# ESPERADO: todos los valores en 0.
chequeos = {
    "premios fuera de 0-6": int(((X[["own_prizes", "opp_prizes"]] < 0) | (X[["own_prizes", "opp_prizes"]] > 6)).any(axis=1).sum()),
    "mazo, mano o banca negativos": int((X[["own_deck", "own_hand", "own_bench_n", "opp_deck", "opp_hand", "opp_bench_n"]] < 0).any(axis=1).sum()),
    "hp activo mayor que max_hp (propio)": int(((X["own_active_present"] == 1) & (X["own_active_hp"] > X["own_active_max_hp"])).sum()),
    "hp activo mayor que max_hp (oponente)": int(((X["opp_active_present"] == 1) & (X["opp_active_hp"] > X["opp_active_max_hp"])).sum()),
}
pd.Series(chequeos)

In [ ]:
# PROPOSITO: medir cuanto predice cada caracteristica por si sola (AUC: 0.5 no sirve, 1.0 perfecto)
# y mostrar las 15 mas fuertes. Es una prueba de fuga: una caracteristica que casi
# adivina el resultado sola suele contener informacion del futuro.
# Se usa una muestra de 200 mil filas para que sea rapida.
# ESPERADO: las mejores entre 0.55 y 0.70 (por ejemplo diferencias de premios, vida y energias).
# ALERTA: cualquier valor sobre 0.85 es sospechoso; revisar de donde sale esa columna.
# Nota: un AUC bajo 0.5 solo indica relacion inversa; la lista ordena por distancia a 0.5.
from sklearn.metrics import roc_auc_score

idx = np.random.RandomState(0).choice(len(X), 200_000, replace=False)
Xs, ys = X.iloc[idx], y.iloc[idx]
auc = pd.Series({c: roc_auc_score(ys, Xs[c]) for c in feats if Xs[c].nunique() > 1})
top = (auc - 0.5).abs().sort_values(ascending=False).head(15).index
print(auc[top].round(3))

In [ ]:
# PROPOSITO: ver como cambia la proporcion de victorias y el numero de filas segun el turno.
# Sirve para decidir mas adelante los cortes de etapa de la partida (inicio, medio, final);
# esos cortes definitivos se calcularan solo con el conjunto de entrenamiento.
# ESPERADO: la columna size baja al avanzar los turnos (pocas partidas llegan a turnos altos).
# La columna mean (prevalencia) debe moverse cerca de 0.5 en los primeros tramos;
# si se aleja mucho en turnos altos, es por pocas partidas muy largas, no por un error.
tramos = pd.cut(states["turn"], [-1, 0, 2, 4, 6, 10, 20, 100])
print(states.groupby(tramos, observed=True)["y"].agg(["size", "mean"]).round(3))